# HyperKT Stage 10 — Frozen Held-Out Test Evaluation

**Final one-time test evaluation.** Runs the 15 **preselected Stage 9E** checkpoints without training, checkpoint selection, or parameter changes. Uses the unchanged Stage 2 test split. Outputs per-seed metrics, student-cluster bootstrap intervals, paired comparisons, prediction artifacts and protocol evidence.

**Required inputs:** `HyperKT_Stage9E-20261102T225711Z-1-001.zip`, `hyperkt_stage2(2).zip`, `hyperkt_stage3b(1).zip`, `hyperkt_stage9g.zip`, `hyperkt_stage9f.zip`. Upload exactly these five ZIPs. **T4/L4 GPU recommended** (CPU works but slower).

**Interpretation:** The DKT baseline is independently implemented; its official-paper fidelity is unverified. All five model seeds share the same held-out student cohort, so seed variation is not a substitute for student-clustered uncertainty. Stage 9F audited training/validation event uniqueness; the test event reconstruction was not independently audited there. Do not tune any model after inspecting test results.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib torch

In [ ]:
import io, os, json, zipfile, hashlib, math, random, tempfile, shutil, pathlib, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, f1_score
from google.colab import files
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)
print('Upload FIVE files: Stage 9E checkpoint ZIP, Stage 2 ZIP, Stage 3B ZIP, Stage 9G ZIP, Stage 9F ZIP')
uploaded=files.upload()
assert len(uploaded)==5, 'Upload exactly five ZIP archives.'
zs={p:zipfile.ZipFile(io.BytesIO(b)) for p,b in uploaded.items()}
for name,z in zs.items(): print(name,len(z.namelist()),'members')
z9=next(z for z in zs.values() if any(x.endswith('/protocol.json') and 'Stage9E' in x for x in z.namelist()))
z2=next(z for z in zs.values() if 'train_sequences.npz' in z.namelist())
z3=next(z for z in zs.values() if 'hypergraph_repaired.npz' in z.namelist())
root=next(x[:-len('protocol.json')] for x in z9.namelist() if x.endswith('/protocol.json'))
protocol=json.loads(z9.read(root+'protocol.json'))
sha2=next(hashlib.sha256(b).hexdigest() for p,b in uploaded.items() if zs[p] is z2)
sha3=next(hashlib.sha256(b).hexdigest() for p,b in uploaded.items() if zs[p] is z3)
assert sha2==protocol['stage2_sha256'], 'Stage 2 ZIP SHA-256 mismatch'
assert sha3==protocol['stage3b_sha256'], 'Stage 3B ZIP SHA-256 mismatch'
print('Source hashes verified')
z9g=next(z for z in zs.values() if 'protocol_lock.json' in z.namelist())
z9f=next(z for z in zs.values() if 'hyperkt_stage9f/event_audit.json' in z.namelist())
lock=json.loads(z9g.read('protocol_lock.json'))
event_audit=json.loads(z9f.read('hyperkt_stage9f/event_audit.json'))
assert lock['checkpoints_verified']==15 and not lock['checkpoint_failures']
assert lock['selected_proposed_model']=='HyperKT_matched' and lock['primary_comparator']=='DKT_matched'
assert event_audit['event_uniqueness_pass'] and event_audit['stage2_sha256']==sha2
assert lock['source_protocol']['stage2_sha256']==sha2 and lock['source_protocol']['stage3b_sha256']==sha3
print('Pre-test lock, checkpoint inventory, and source hashes verified.')


In [ ]:
def read_npz(z,name):
    with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a: return {k:a[k].copy() for k in a.files}
train=read_npz(z2,'train_sequences.npz'); val=read_npz(z2,'val_sequences.npz'); test=read_npz(z2,'test_sequences.npz')
struct=read_npz(z3,'hypergraph_repaired.npz')
manifest=json.loads(z2.read('stage2_report.json'))
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),test['target_skill'].max(),struct['skill_ids'].max())+1)
H=np.zeros((N_SKILLS,struct['assignments'].shape[1]),dtype=np.float32)
H[struct['skill_ids'].astype(int)]=struct['assignments'].astype(np.float32)
MAX_LEN=train['history_skill'].shape[1]
print('N_SKILLS:',N_SKILLS,'MAX_LEN:',MAX_LEN,'hyperedges:',H.shape[1])
assert N_SKILLS==151 and MAX_LEN==200, 'Unexpected source configuration: review before running.'

assert len(test['mask'])==484 and int(test['mask'].sum())==32522
assert not (set(test['user_id']) & set(train['user_id'])) and not (set(test['user_id']) & set(val['user_id']))
print('Test windows:',len(test['mask']),'targets:',int(test['mask'].sum()),'students:',len(set(test['user_id'])))


In [ ]:
class SeqDataset(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self):return len(self.d['mask'])
    def __getitem__(self,i):
        return tuple(torch.as_tensor(self.d[k][i],dtype=torch.long if k in ('history_skill','history_correct','target_skill') else torch.float32) for k in ('history_skill','history_correct','target_skill','target_correct','mask'))
TRAIN_DS=SeqDataset(train); VAL_DS=SeqDataset(val); TEST_DS=SeqDataset(test)
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    try:torch.use_deterministic_algorithms(True,warn_only=True)
    except Exception:pass
class CausalBlock(nn.Module):
    def __init__(self,d=128,heads=8,drop=.1,decay=False):
        super().__init__();self.heads=heads;self.dhead=d//heads;self.decay=decay
        self.qkv=nn.Linear(d,3*d);self.proj=nn.Linear(d,d);self.drop=nn.Dropout(drop)
        self.ln1=nn.LayerNorm(d);self.ln2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d,d))
        if decay:self.log_rate=nn.Parameter(torch.full((heads,),-3.0))
    def forward(self,x):
        B,T,D=x.shape; z=self.ln1(x)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        def split(t):return t.reshape(B,T,self.heads,self.dhead).transpose(1,2)
        q,k,v=map(split,(q,k,v))
        logits=torch.matmul(q,k.transpose(-2,-1))*(self.dhead**-.5)
        pos=torch.arange(T,device=x.device)
        dist=(pos[:,None]-pos[None,:]).clamp(min=0).float()
        if self.decay:logits=logits-torch.nn.functional.softplus(self.log_rate)[None,:,None,None]*dist[None,None,:,:]
        future=pos[None,:]>pos[:,None]
        logits=logits.masked_fill(future[None,None,:,:],torch.finfo(logits.dtype).min)
        a=torch.softmax(logits,dim=-1)
        x=x+self.drop(self.proj(torch.matmul(self.drop(a),v).transpose(1,2).reshape(B,T,D)))
        x=x+self.drop(self.ff(self.ln2(x)))
        return x
class HyperKT(nn.Module):
    def __init__(self,n_skills,H,hyper=False,decay=False,d=128,layers=2,heads=8,drop=.1):
        super().__init__();self.hyper=hyper
        self.skill=nn.Embedding(n_skills,d,padding_idx=1)
        self.response=nn.Embedding(2,d)
        self.pos=nn.Embedding(MAX_LEN,d)
        self.blocks=nn.ModuleList([CausalBlock(d,heads,drop,decay) for _ in range(layers)])
        self.norm=nn.LayerNorm(d)
        if hyper:
            self.register_buffer('assignments',torch.tensor(H))
            self.edge_embedding=nn.Parameter(torch.randn(H.shape[1],d)*.02)
            self.fusion=nn.Linear(d*2,d)
        self.output=nn.Sequential(nn.LayerNorm(d*2),nn.Linear(d*2,d),nn.GELU(),nn.Dropout(drop),nn.Linear(d,1))
    def forward(self,hs,hc,ts):
        B,T=hs.shape; pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x)
        target=self.skill(ts)
        if self.hyper:
            structural=self.assignments[ts]@self.edge_embedding
            target=target+torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)


In [ ]:
class DKT(nn.Module):
    # Independent Stage 9C baseline implementation, not a certified official DKT reproduction.
    def __init__(self,n,d=128,dropout=.1):
        super().__init__();self.n=n;self.emb=nn.Embedding(n*2,d)
        self.rnn=nn.LSTM(d,d,batch_first=True);self.drop=nn.Dropout(dropout);self.out=nn.Linear(d,n)
    def forward(self,hs,hc,ts):
        h,_=self.rnn(self.emb(hs+hc.clamp(0,1)*self.n))
        return self.out(self.drop(h)).gather(-1,ts.unsqueeze(-1)).squeeze(-1)

class HyperKTGated(HyperKT):
    # Only change vs original HyperKT: a scalar learned gate on the hypergraph residual.
    def __init__(self,n_skills,H,d=128,layers=2,heads=8,drop=.1):
        super().__init__(n_skills,H,hyper=True,decay=True,d=d,layers=layers,heads=heads,drop=drop)
        self.hyper_gate=nn.Parameter(torch.tensor(-2.0))
    def forward(self,hs,hc,ts):
        B,T=hs.shape;pos=torch.arange(T,device=hs.device)[None,:]
        x=self.skill(hs)+self.response(hc.clamp(0,1))+self.pos(pos)
        for layer in self.blocks:x=layer(x)
        x=self.norm(x);target=self.skill(ts)
        structural=self.assignments[ts]@self.edge_embedding
        target=target+torch.sigmoid(self.hyper_gate)*torch.tanh(self.fusion(torch.cat([target,structural],dim=-1)))
        return self.output(torch.cat([x,target],dim=-1)).squeeze(-1)

def make_model(name):
    if name=='DKT_matched':return DKT(N_SKILLS).to(DEVICE)
    if name=='HyperKT_matched':return HyperKT(N_SKILLS,H,hyper=True,decay=True).to(DEVICE)
    if name=='HyperKT_gated':return HyperKTGated(N_SKILLS,H).to(DEVICE)
    raise ValueError(name)



In [ ]:
CFG=protocol['config']; SEEDS=protocol['seeds']; VARIANTS=protocol['models']
assert SEEDS==[42,123,2024,3407,7777]
assert set(VARIANTS)=={'DKT_matched','HyperKT_matched','HyperKT_gated'}
assert len(test['mask'])==484 and int(test['mask'].sum())==32522
print('Protocol:',json.dumps(protocol,indent=2))

In [ ]:
def metrics(y,p):
    y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    edges=np.linspace(0,1,16);inds=np.minimum(np.digitize(p,edges[1:-1]),14)
    ece=sum((inds==i).mean()*abs(y[inds==i].mean()-p[inds==i].mean()) for i in range(15) if np.any(inds==i))
    return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5,zero_division=0)),ece15=float(ece),n=int(len(y)))
def loader(ds,shuffle,seed):
    g=torch.Generator().manual_seed(seed)
    return DataLoader(ds,batch_size=CFG['batch_size'],shuffle=shuffle,generator=g,num_workers=0,pin_memory=True)
def evaluate(model,dl):
    model.eval();Y=[];P=[]
    with torch.no_grad():
        for hs,hc,ts,y,mask in dl:
            hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
            p=torch.sigmoid(model(hs,hc,ts)).cpu().numpy(); m=mask.numpy().astype(bool)
            Y.append(y.numpy()[m]);P.append(p[m])
    return metrics(np.concatenate(Y),np.concatenate(P))


## One-time frozen inference

This cell reads all 15 checkpoint weights, verifies their hashes against Stage 9G, and computes test predictions. **Do not rerun after changing model definitions, checkpoints, or evaluation settings.** Predictions are saved for auditing, not model selection.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from scipy.stats import t
from collections import defaultdict
out=Path('/content/hyperkt_stage10');out.mkdir(exist_ok=True)
verified=pd.read_csv(io.BytesIO(z9g.read('checkpoint_verification.csv')))
assert len(verified)==15
TEST_LOADER=loader(TEST_DS,False,42)

def infer_full(model,dl):
    model.eval();ys=[];ps=[];uids=[]
    offset=0
    with torch.inference_mode():
        for hs,hc,ts,y,mask in dl:
            bs=hs.shape[0];hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
            pred=torch.sigmoid(model(hs,hc,ts)).cpu().numpy()
            m=mask.numpy().astype(bool);truth=y.numpy()
            ys.append(truth[m].astype(np.int8));ps.append(pred[m].astype(np.float32))
            row_users=np.asarray(test['user_id'][offset:offset+bs]).astype(str)
            uids.append(np.repeat(row_users, m.sum(axis=1)))
            offset+=bs
    assert offset==len(test['mask'])
    return np.concatenate(ys),np.concatenate(ps),np.concatenate(uids)

records=[]; predictions={}; checkpoint_hashes=[]
for name in VARIANTS:
    for seed in SEEDS:
        key=f'{root}{name}_seed{seed}/best.pt'
        raw=z9.read(key); digest=hashlib.sha256(raw).hexdigest()
        v=verified.loc[(verified.variant==name)&(verified.seed==seed)]
        assert len(v)==1 and v.iloc[0]['sha256']==digest, f'Checkpoint hash mismatch: {name}, {seed}'
        ckpt=torch.load(io.BytesIO(raw),map_location='cpu',weights_only=True)
        assert ckpt['variant']==name and ckpt['seed']==seed and ckpt['config']==CFG
        assert ckpt['stage2_sha256']==sha2 and ckpt['stage3b_sha256']==sha3
        model=make_model(name);model.load_state_dict(ckpt['state_dict'],strict=True)
        y,p,u=infer_full(model,TEST_LOADER)
        assert len(y)==32522 and len(np.unique(u))>0
        key2=f'{name}_seed{seed}'
        predictions[key2]={'y':y,'p':p,'user_id':u}
        m=metrics(y,p)
        m.update(accuracy=float(accuracy_score(y,p>=.5)),precision=float(precision_score(y,p>=.5,zero_division=0)),recall=float(recall_score(y,p>=.5,zero_division=0)))
        records.append({'variant':name,'seed':seed,'checkpoint_sha256':digest,**m,'n_students':len(np.unique(u))})
        checkpoint_hashes.append({'variant':name,'seed':seed,'sha256':digest})
        np.savez_compressed(out/f'predictions_{key2}.npz',y=y,p=p,user_id=u)
        print(name,seed,'test AUC',round(m['auc'],6))
        del model
results=pd.DataFrame(records)
assert len(results)==15
results.to_csv(out/'per_seed_test_metrics.csv',index=False)
pd.DataFrame(checkpoint_hashes).to_csv(out/'checkpoint_hashes.csv',index=False)
display(results)


## Aggregation and student-clustered uncertainty

Student bootstrap resamples **students** (not individual events) with replacement. Intervals below describe cohort uncertainty conditional on each frozen checkpoint. We additionally report seed-level standard deviations. The five seeds and same test cohort do not form independent external replications.

In [ ]:
metric_names=['auc','nll','brier','f1','ece15','accuracy','precision','recall']
summary=results.groupby('variant')[metric_names].agg(['mean','std'])
summary.to_csv(out/'five_seed_test_summary.csv')
display(summary)

def cluster_bootstrap_metric(y,p,u, metric='auc',B=1000,seed=20261103):
    rng=np.random.default_rng(seed)
    ids,inv=np.unique(u,return_inverse=True)
    groups=[np.flatnonzero(inv==i) for i in range(len(ids))]
    vals=[]
    for _ in range(B):
        chosen=rng.integers(0,len(ids),size=len(ids))
        ix=np.concatenate([groups[i] for i in chosen])
        if metric=='auc':
            if len(np.unique(y[ix]))<2:continue
            vals.append(roc_auc_score(y[ix],p[ix]))
        elif metric=='brier':vals.append(brier_score_loss(y[ix],p[ix]))
        else:raise ValueError(metric)
    return np.quantile(vals,[.025,.975]).tolist(),len(vals)

cis=[]
for name in VARIANTS:
    for seed in SEEDS:
        a=predictions[f'{name}_seed{seed}']
        ci,nrep=cluster_bootstrap_metric(a['y'],a['p'],a['user_id'],B=1000,seed=seed+10000)
        cis.append({'variant':name,'seed':seed,'auc_ci_low':ci[0],'auc_ci_high':ci[1],'bootstrap_reps':nrep})
pd.DataFrame(cis).to_csv(out/'student_cluster_auc_intervals.csv',index=False)
display(pd.DataFrame(cis))


## Paired comparison against DKT

For each seed, the DKT and HyperKT models predict the **same ordered target events**. We compute paired seed-level AUC differences and a student-cluster bootstrap of the **difference** for each seed. These are descriptive uncertainty summaries, not a guarantee of cross-dataset generalization.

In [ ]:
comparisons=[]
for other in ['HyperKT_matched','HyperKT_gated']:
    seed_diffs=[]
    for seed in SEEDS:
        d=predictions[f'DKT_matched_seed{seed}']; h=predictions[f'{other}_seed{seed}']
        assert np.array_equal(d['y'],h['y']) and np.array_equal(d['user_id'],h['user_id'])
        delta=roc_auc_score(d['y'],d['p'])-roc_auc_score(h['y'],h['p'])
        seed_diffs.append(delta)
        rng=np.random.default_rng(seed+19000)
        ids,inv=np.unique(d['user_id'],return_inverse=True)
        groups=[np.flatnonzero(inv==i) for i in range(len(ids))]
        boot=[]
        for _ in range(1000):
            ix=np.concatenate([groups[i] for i in rng.integers(len(ids),size=len(ids))])
            if len(np.unique(d['y'][ix]))<2:continue
            boot.append(roc_auc_score(d['y'][ix],d['p'][ix])-roc_auc_score(d['y'][ix],h['p'][ix]))
        lo,hi=np.quantile(boot,[.025,.975])
        comparisons.append({'comparator':other,'seed':seed,'dkt_minus_comparator_auc':delta,'student_boot_ci_low':lo,'student_boot_ci_high':hi})
    print(other,'mean paired AUC difference',np.mean(seed_diffs),'SD',np.std(seed_diffs,ddof=1))
pd.DataFrame(comparisons).to_csv(out/'paired_auc_comparisons.csv',index=False)
display(pd.DataFrame(comparisons))


## Reproducibility record, figure, and export

Results must be reported **as observed**, including any unfavorable comparisons. This notebook does not modify model parameters or choose a winner from test results.

In [ ]:
import matplotlib.pyplot as plt, platform
fig,ax=plt.subplots(figsize=(8,4.5))
order=['DKT_matched','HyperKT_matched','HyperKT_gated']
means=[results.loc[results.variant==x,'auc'].mean() for x in order]
sds=[results.loc[results.variant==x,'auc'].std(ddof=1) for x in order]
ax.bar(order,means,yerr=sds,capsize=5)
ax.set_ylabel('Held-out test ROC-AUC');ax.set_title('HyperKT Stage 10: five-seed frozen evaluation')
ax.set_ylim(max(0,min(means)-.04),min(1,max(means)+.025))
plt.xticks(rotation=12,ha='right');fig.tight_layout();fig.savefig(out/'test_auc_comparison.png',dpi=200);plt.show()
record={'stage':'10','one_time_held_out_test':True,'model_selection_on_test':False,'checkpoint_count':len(records),
 'seeds':SEEDS,'models':VARIANTS,'stage2_sha256':sha2,'stage3b_sha256':sha3,
 'stage9g_lock':lock,'stage9f_event_integrity':event_audit['event_uniqueness_pass'],
 'test_event_uniqueness_independently_reconstructed':False,
 'baseline_fidelity':'Independent DKT implementation; original published implementation not independently verified',
 'hypergraph_fixed_across_seeds':True,'student_bootstrap_reps':1000,
 'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__,'device':str(DEVICE)}
(out/'evaluation_protocol.json').write_text(json.dumps(record,indent=2,default=str))
(out/'README.txt').write_text('Frozen Stage 10 test evaluation. No retraining. 15 checkpoints, 5 seeds per architecture. DKT independently implemented. Test results must not be used for further tuning. Student-cluster bootstrap is conditional on fixed checkpoints. Test event uniqueness not independently reconstructed.\n')
zipout=Path('/content/hyperkt_stage10.zip')
with zipfile.ZipFile(zipout,'w',zipfile.ZIP_DEFLATED) as zz:
    for f in sorted(out.iterdir()):zz.write(f,f.name)
print('Created',zipout, 'size (MB):',round(zipout.stat().st_size/1e6,2))
files.download(str(zipout))
